# 01 API Smoke Test
Assumption: API is running at `http://127.0.0.1:8000`.
This notebook covers create user -> alias search/select log -> summary -> optimize -> infeasible check.

In [3]:
import json
from datetime import datetime, timezone
from urllib.request import Request, urlopen
from urllib.error import HTTPError

BASE_URL = "http://127.0.0.1:8000"

def api(method, path, payload=None):
    url = f"{BASE_URL}{path}"
    body = None
    headers = {"Accept": "application/json"}
    if payload is not None:
        body = json.dumps(payload).encode("utf-8")
        headers["Content-Type"] = "application/json"
    req = Request(url, data=body, headers=headers, method=method.upper())
    try:
        with urlopen(req, timeout=20) as resp:
            raw = resp.read().decode("utf-8")
            return resp.status, (json.loads(raw) if raw else {})
    except HTTPError as exc:
        raw = exc.read().decode("utf-8")
        try:
            payload = json.loads(raw)
        except Exception:
            payload = {"raw": raw}
        return exc.code, payload

def show(title, obj):
    print(f"\\n=== {title} ===")
    print(json.dumps(obj, indent=2, ensure_ascii=False))

In [4]:
# 1) Create user
status, user = api("POST", "/v1/users", {"name": "smoke-user"})
assert status == 200, (status, user)
user_id = user["user_id"]
show("Create user", user)

\n=== Create user ===
{
  "status": "ok",
  "user_id": "cb8100d3-f979-44d7-8235-61c2eb65f17e"
}


In [5]:
# 2) Set profile + derive targets + set config
status, profile_resp = api("POST", "/v1/profile", {
    "user_id": user_id,
    "age": 30, "sex": "male", "height_cm": 195, "weight_kg": 109,
    "activity_level": "moderate", "goal": "maintain"
})
assert status == 200, (status, profile_resp)

status, derived = api("POST", "/v1/profile/derive-targets", {"user_id": user_id, "strictness": "normal"})
assert status == 200, (status, derived)
targets = derived["targets"]

status, cfg = api("PUT", "/v1/config", {
    "user_id": user_id,
    "config": {
        "horizon_days": 1,
        "constraints": {
            "calories_kcal": {"min": 700, "max": 1100},
            "protein_g": {"min": 60, "max": 130},
            "carbs_g": {"min": 50, "max": 200},
            "fat_g": {"min": 15, "max": 70},
            "fiber_g": {"min": 8},
            "sat_fat_g": {"max": 20},
            "sodium_mg": {"max": 3000},
            "budget_try": {"max": 250}
        },
        "objectives_lex": [
            {
                "name": "min_total_deviation",
                "type": "deviation",
                "targets": ["calories_kcal", "protein_g", "carbs_g", "fat_g"],
                "target_values": {
                    "calories_kcal": targets["calories_kcal"],
                    "protein_g": targets["protein_g"],
                    "carbs_g": targets["carbs_g"],
                    "fat_g": targets["fat_g"]
                },
                "tolerance": 0.01
            },
            {"name": "min_cost", "type": "linear", "metric": "cost_try", "sense": "min"}
        ],
        "food_bounds": {"min_grams_per_food": 0, "max_grams_per_food": 400}
    }
})
assert status == 200, (status, cfg)
show("Derived targets", derived)
show("Set config", cfg)

\n=== Derived targets ===
{
  "targets": {
    "calories_kcal": 3353.81,
    "protein_g": 174.4,
    "carbs_g": 443.33,
    "fat_g": 98.1,
    "fiber_g": 46.95,
    "sat_fat_g": 37.26,
    "sodium_mg": 2300.0,
    "budget_try": 335.38
  },
  "constraints": {
    "calories_kcal": {
      "min": 3186.1195,
      "max": 3521.5005
    },
    "protein_g": {
      "min": 165.68,
      "max": 183.12
    },
    "carbs_g": {
      "min": 421.16349999999994,
      "max": 465.4965
    },
    "fat_g": {
      "min": 93.195,
      "max": 103.005
    },
    "fiber_g": {
      "min": 46.95,
      "max": null
    },
    "sat_fat_g": {
      "min": null,
      "max": 37.26
    },
    "sodium_mg": {
      "min": null,
      "max": 2300.0
    },
    "budget_try": {
      "min": null,
      "max": 335.38
    }
  }
}
\n=== Set config ===
{
  "status": "ok"
}


In [8]:
# 3) logs/search with alias term: simit
status, found = api("POST", "/v1/logs/search", {"query": "simit", "limit": 5, "provider": "openfoodfacts"})
assert status == 200, (status, found)
assert found["candidates"], "No candidates returned"
show("logs/search simit", found)

# 4) Pick first if selection_required, otherwise recommended
if found.get("selection_required", False):
    chosen_food_id = found["candidates"][0]["food_id"]
else:
    chosen_food_id = found.get("recommended_food_id") or found["candidates"][0]["food_id"]
print("Chosen food_id:", chosen_food_id)

\n=== logs/search simit ===
{
  "candidates": [
    {
      "food_id": "simit",
      "name": "Simit",
      "source": "local",
      "source_id": null,
      "match_type": "alias",
      "matched_alias": "simit",
      "calories_kcal_g": 2.72,
      "protein_g_g": 0.08,
      "carbs_g_g": 0.53,
      "fat_g_g": 0.07,
      "fiber_g_g": 0.03,
      "sat_fat_g_g": 0.012,
      "sodium_mg_g": 4.1
    }
  ],
  "selection_required": false,
  "recommended_food_id": "simit",
  "search_id": "221b5f4c-f2f5-48e2-936c-6c77e1c41f19",
  "expires_at": "2026-02-28T15:14:05.112303Z"
}
Chosen food_id: simit


In [9]:
# 5) Log grams with chosen food_id
status, logged = api("POST", "/v1/logs/select", {
    "user_id": user_id,
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "food_id": chosen_food_id,
    "grams": 120.0
})
assert status == 200, (status, logged)
show("logs/select", logged)

\n=== logs/select ===
{
  "status": "ok",
  "logged": {
    "food_id": "simit",
    "food_name": "Simit",
    "grams": 120.0,
    "timestamp": "2026-02-28T14:44:22.709187Z"
  }
}


In [10]:
# 6) Summary today
status, summary = api("GET", f"/v1/summary/today?user_id={user_id}")
assert status == 200, (status, summary)
show("summary/today", summary)

\n=== summary/today ===
{
  "consumed": {
    "calories_kcal": 326.40000000000003,
    "protein_g": 9.6,
    "carbs_g": 63.6,
    "fat_g": 8.4,
    "fiber_g": 3.5999999999999996,
    "sat_fat_g": 1.44,
    "sodium_mg": 491.99999999999994,
    "cost_try": 7.199999999999999,
    "preference_score": 19.2
  },
  "remaining_bounds": {
    "calories_kcal": {
      "min": 373.59999999999997,
      "max": 773.5999999999999
    },
    "protein_g": {
      "min": 50.4,
      "max": 120.4
    },
    "carbs_g": {
      "min": 0.0,
      "max": 136.4
    },
    "fat_g": {
      "min": 6.6,
      "max": 61.6
    },
    "fiber_g": {
      "min": 4.4,
      "max": null
    },
    "sat_fat_g": {
      "min": null,
      "max": 18.56
    },
    "sodium_mg": {
      "min": null,
      "max": 2508.0
    },
    "budget_try": {
      "min": null,
      "max": 250.0
    }
  }
}


In [11]:
# 7) Optimize plan
status, plan = api("POST", "/v1/plan/optimize", {"user_id": user_id, "horizon_days": 1, "meal_slots": ["lunch", "dinner", "snack"]})
assert status == 200, (status, plan)
show("plan/optimize", plan)
print("Summary:", plan.get("today_plan_summary", ""))

\n=== plan/optimize ===
{
  "plan": [
    {
      "food_id": "oats_dry",
      "food_name": "Oats Dry",
      "grams": 169.28463367192563
    },
    {
      "food_id": "chicken_breast",
      "food_name": "Chicken Breast",
      "grams": 69.74713637346014
    }
  ],
  "totals_planned": {
    "calories_kcal": 773.5999999999999,
    "protein_g": 50.400000000000006,
    "carbs_g": 111.72785822347092,
    "fat_g": 14.36082126647936,
    "fiber_g": 16.928463367192563,
    "sat_fat_g": 2.728886967797709,
    "sodium_mg": 54.998573589799015,
    "cost_try": 21.018716230819106,
    "preference_score": 44.42066133563864
  },
  "constraint_report": [
    {
      "name": "calories_kcal_min",
      "status": "ok",
      "slack": 399.99999999999994
    },
    {
      "name": "calories_kcal_max",
      "status": "binding",
      "slack": 0.0
    },
    {
      "name": "protein_g_min",
      "status": "binding",
      "slack": 7.105427357601002e-15
    },
    {
      "name": "protein_g_max",
      "s

In [12]:
# 8) Force infeasible config and verify 422 + suggested_relaxations
status, _ = api("PUT", "/v1/config", {
    "user_id": user_id,
    "config": {
        "horizon_days": 1,
        "constraints": {"protein_g": {"min": 1000.0}, "calories_kcal": {"max": 150.0}},
        "objectives_lex": [
            {"name": "min_total_deviation", "type": "deviation", "targets": ["protein_g", "calories_kcal"], "target_values": {"protein_g": 1000.0, "calories_kcal": 100.0}}
        ],
        "food_bounds": {"min_grams_per_food": 0.0, "max_grams_per_food": 50.0}
    }
})
assert status == 200

status, inf = api("POST", "/v1/plan/optimize", {"user_id": user_id, "horizon_days": 1})
assert status == 422, (status, inf)
detail = inf.get("detail", {})
assert detail.get("code") == "infeasible_plan", detail
assert detail.get("suggested_relaxations"), detail
show("Infeasible 422 response", inf)

\n=== Infeasible 422 response ===
{
  "detail": {
    "code": "infeasible_plan",
    "summary": "Optimization infeasible for the current remaining bounds and food limits.",
    "stage": 1,
    "objective": "min_total_deviation",
    "solver_message": "The problem is infeasible. (HiGHS Status 8: model_status is Infeasible; primal_status is None)",
    "suggested_relaxations": [
      {
        "constraint": "protein_g",
        "issue": "min_unreachable",
        "current_min": 990.4,
        "current_max": null,
        "recommended_min": 42.7,
        "recommended_max": null,
        "delta": 947.6999999999999,
        "reason": "Minimum required total is above best-case capacity under current food bounds."
      },
      {
        "constraint": "calories_kcal",
        "issue": "already_exceeded_max",
        "current_min": null,
        "current_max": -176.40000000000003,
        "recommended_min": null,
        "recommended_max": 0.0,
        "delta": 176.40000000000003,
        "r